# Gap-Ratio Benchmark for a Random-Field XXZ Chain

This notebook replaces the old `GapRatioXXZ.jl` script. It shows a standard exact-diagonalization workflow for level statistics in a symmetry sector: construct the Hamiltonian, diagonalize it, and compute the mean adjacent gap ratio.

In [ ]:
import Pkg

function find_repo_root(start = pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "EDKit.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate EDKit.jl repo root from $(start)")
        dir = parent
    end
end

const DEV = true
const REPO_ROOT = find_repo_root()
Pkg.activate(REPO_ROOT)
if DEV
    pushfirst!(LOAD_PATH, REPO_ROOT)
end

using EDKit
using LinearAlgebra
using Statistics

using Random
Random.seed!(7)


## Build the disorder realization

We work in the zero-magnetization sector of a length-10 XXZ chain with random longitudinal fields. The point is not to make a definitive many-body-localization study at this size; it is to show the workflow and the basic diagnostic.

In [ ]:
L = 10
delta = 1.2
disorder = 0.8
fields = disorder .* randn(L)

sector = basis(L = L, N = L ÷ 2)
exchange = spin((1.0, "xx"), (1.0, "yy"), (delta, "zz"))

H = trans_inv_operator(exchange, 2, sector)
H += operator([h * spin("Z") for h in fields], collect(1:L), sector)

vals = sort(eigvals(Hermitian(H)))
ratios = gapratio(vals)
summary = (
    sector_dimension = size(sector, 1),
    mean_gap_ratio = meangapratio(vals),
    central_eigenvalues = vals[length(vals) ÷ 2 - 2:length(vals) ÷ 2 + 2],
)
summary


## Plot the gap-ratio distribution

For small systems, the full distribution is often more informative than just the mean. Even when the physics is not yet in a clean asymptotic regime, the histogram still tells you whether the spacings are clustering or repelling.

In [ ]:
if isdefined(Main, :IJulia)
    using Plots
    histogram(ratios; bins = 16, normalize = :pdf, xlabel = "r", ylabel = "pdf", label = "sample", title = "Adjacent gap-ratio distribution")
    vline!([mean(ratios)]; label = "mean")
else
    @info "Skipping Plots outside IJulia; returning raw ratios instead"
    ratios
end


## Takeaway

This is one of the most common ED workflows in the package: use `basis(...)` to reduce the Hilbert space, construct the operator once, diagonalize it, and then post-process the eigenvalues into a diagnostic of interest. At larger sizes the symmetry reduction matters as much as the statistic itself.